# ADH Assignment 1: 2SLS Regressions (Jupyter Version)

This notebook replicates **ADH Assignment 1** (originally written in Stata as `ADH_Assignment1.do`).
It performs 20 weighted 2SLS regressions using `workfile_china.dta`.

### Required Packages:
- `pandas`
- `linearmodels`


In [1]:
# Setup & Imports
import sys
from pathlib import Path
import pandas as pd
from linearmodels.iv import IV2SLS


## Define Constants & Paths
Update `DATA_PATH` below if your `workfile_china.dta` file is located elsewhere.

In [2]:
# Set dataset path
DATA_PATH = Path('workfile_china.dta')  # Adjust path if needed

OUTCOMES = [
    'd_sh_empl_mfg',
    'd_sh_empl_nmfg',
    'd_sh_unempl',
    'd_sh_nilf',
    'd_avg_lnwkwage',
]

CONTROLS = [
    'l_shind_manuf_cbp',
    'l_sh_popedu_c',
    'l_sh_popfborn',
    'l_sh_empl_f',
    'l_sh_routine33',
    'l_task_outsource',
]


## Data Preparation & Variable Construction
- Reads `workfile_china.dta`.
- Constructs region indicator controls (`reg*`).
- Constructs CZ-level 1990 manufacturing share (`mshare90`).
- Builds year-interaction terms (`i.yr#c.mshare90`).

In [3]:
# Load Data
df = pd.read_stata(DATA_PATH, convert_categoricals=False)

# Region controls
regions = [col for col in df.columns if col.startswith('reg')]
controls = CONTROLS + regions
df['_cons'] = 1.0

# Replicate Stata: bys cz: egen mshare90=max(l_shind_manuf_cbp*(yr==1990))
df['mshare90'] = (
    (df['l_shind_manuf_cbp'] * (df['yr'] == 1990))
    .groupby(df['czone'], dropna=False)
    .transform('max')
)
df['mshare90'] = df['mshare90'].astype('float32').astype('float64')

# Create year interaction terms (i.yr#c.mshare90)
interactions = []
for year in sorted(df['yr'].dropna().unique()):
    name = f'{int(year)}.yr#c.mshare90'
    df[name] = df['mshare90'] * (df['yr'] == year)
    interactions.append(name)

print(f'Dataset loaded successfully. Total rows: {len(df)}')


Dataset loaded successfully. Total rows: 1444


## Estimation Helper Function
Defines 2SLS estimation using `linearmodels.iv.IV2SLS` matching Stata's `ivregress 2sls ... [aw=timepwt48], cluster(statefip)`.

In [4]:
results = []

def estimate(outcome, sample, exog, label, display=True):
    columns = [outcome, 'd_tradeusch_pw', 'd_tradeotch_pw_lag', 'timepwt48', 'statefip'] + exog + ['_cons']
    data = sample[columns].dropna()
    data = data.loc[data['timepwt48'] > 0]
    
    result = IV2SLS(
        dependent=data[outcome],
        exog=data[exog + ['_cons']],
        endog=data[['d_tradeusch_pw']],
        instruments=data[['d_tradeotch_pw_lag']],
        weights=data['timepwt48'],
    ).fit(cov_type='clustered', clusters=data['statefip'], debiased=False)
    
    results.append((label, outcome, result))
    if display:
        print(f'\n=== {label}: {outcome} ===\n')
        print(result.summary)
    return result


## 1. Single-Period Regressions (Unstacked)
- **1990–2000**
- **2000–2007**

In [5]:
for year, label in [(1990, '1990–2000'), (2000, '2000–2007')]:
    print(f'\n============================== {label} ==============================')
    for outcome in OUTCOMES:
        estimate(outcome, df.loc[df['yr'] == year], controls, label)



============================== 1990–2000 ==============================

=== 1990–2000: d_sh_empl_mfg ===

                          IV-2SLS Estimation Summary                          
Dep. Variable:          d_sh_empl_mfg   R-squared:                      0.4856
Estimator:                    IV-2SLS   Adj. R-squared:                 0.4747
No. Observations:                 722   F-statistic:                    750.71
Date:                Tue, Oct 27 2026   P-value (F-stat)                0.0000
Time:                        11:05:13   Distribution:                 chi2(15)
Cov. Estimator:             clustered                                         
                                                                              
                                 Parameter Estimates                                 
                   Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------------------------------------------------------------------------
l_

## 2. Stacked Regressions
- **Baseline Stacked Specification**
- **Stacked with Year-Specific 1990 Manufacturing-Share Slopes**

In [6]:
print('\n============================== Stacked Regressions ==============================')
for outcome in OUTCOMES:
    # Stacked baseline
    estimate(outcome, df, controls + ['t2'], 'Stacked baseline')
    
    # Stacked with year-specific 1990 manufacturing share slopes
    varying_controls = (
        CONTROLS + regions if outcome == 'd_avg_lnwkwage'
        else CONTROLS[:1] + regions + CONTROLS[1:]
    )
    estimate(outcome, df, interactions + varying_controls + ['t2'],
             'Stacked with year-specific 1990 manufacturing-share slopes')



============================== Stacked Regressions ==============================

=== Stacked baseline: d_sh_empl_mfg ===

                          IV-2SLS Estimation Summary                          
Dep. Variable:          d_sh_empl_mfg   R-squared:                      0.3429
Estimator:                    IV-2SLS   Adj. R-squared:                 0.3355
No. Observations:                1444   F-statistic:                    702.76
Date:                Tue, Oct 27 2026   P-value (F-stat)                0.0000
Time:                        11:05:21   Distribution:                 chi2(16)
Cov. Estimator:             clustered                                         
                                                                              
                                 Parameter Estimates                                 
                   Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-----------------------------------------------------------------------